# Task 1 — Movie Genre Classification
### SCT Machine Learning Internship

**Objective:** Predict a movie's genre from its plot summary using Natural Language Processing (NLP).

**Workflow:** Load dataset → Explore data → Clean text → TF-IDF vectorization → Train/validate classifier → Evaluate → Save model and results.

> Dataset expected at `../data/train_data.txt`. The commonly used IMDb Genre Classification dataset stores fields separated by `:::`: ID, title, genre, description.


In [ ]:
# 1. Imports and configuration
from pathlib import Path
import re
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, classification_report, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_DIR / "data" / "train_data.txt"
RESULTS_DIR = PROJECT_DIR / "results"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
RESULTS_DIR.mkdir(exist_ok=True)
ARTIFACTS_DIR.mkdir(exist_ok=True)

print("Project directory:", PROJECT_DIR.resolve())
print("Dataset path:", DATA_PATH.resolve())


## 2. Load the dataset

Download `train_data.txt` from the dataset link in the repository README and place it in the `data/` directory. Do not commit the dataset unless its license and terms allow it.


In [ ]:
# 2. Load dataset
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Download train_data.txt and place it in the data/ folder. "
        "See README.md for the dataset source."
    )

df = pd.read_csv(
    DATA_PATH,
    sep=r"\s*:::\s*",
    engine="python",
    header=None,
    names=["movie_id", "title", "genre", "plot"],
    on_bad_lines="skip",
    encoding="utf-8"
)

df = df.dropna(subset=["genre", "plot"]).copy()
for column in ["title", "genre", "plot"]:
    df[column] = df[column].astype(str).str.strip()
df = df[(df["genre"] != "") & (df["plot"] != "")]
df = df.drop_duplicates(subset=["plot", "genre"]).reset_index(drop=True)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())


## 3. Exploratory data analysis

Check class balance before training. Genre classification can be challenging when some genres have far fewer examples than others.


In [ ]:
# 3. Dataset overview and genre distribution
print("Number of genres:", df["genre"].nunique())
print("Missing values:\n", df[["title", "genre", "plot"]].isna().sum())
print("\nTop 20 genres:")
display(df["genre"].value_counts().head(20).rename("count").to_frame())

genre_counts = df["genre"].value_counts().head(20).sort_values()
plt.figure(figsize=(10, 7))
genre_counts.plot(kind="barh")
plt.title("20 Most Frequent Movie Genres")
plt.xlabel("Number of movies")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()


## 4. Prepare training and validation sets

The model uses the plot summary as input (`X`) and the provided genre as the target (`y`). A held-out validation set estimates how well the model performs on unseen examples.


In [ ]:
# 4. Split the data
X = df["plot"]
y = df["genre"]

# Stratification preserves genre proportions when every class has at least 2 examples.
stratify_target = y if y.value_counts().min() >= 2 else None
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=stratify_target
)

print(f"Training examples: {len(X_train):,}")
print(f"Validation examples: {len(X_valid):,}")


## 5. Build and train the model

**TF-IDF** converts text into numerical features. Word n-grams represent words and short phrases; character n-grams capture useful text fragments and spelling patterns. **LinearSVC** is a strong baseline for high-dimensional text classification.


In [ ]:
# 5. TF-IDF + Linear Support Vector Classifier
text_features = FeatureUnion([
    ("word_tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words="english",
        ngram_range=(1, 2),
        max_features=180_000,
        sublinear_tf=True,
        min_df=2
    )),
    ("char_tfidf", TfidfVectorizer(
        analyzer="char_wb",
        lowercase=True,
        ngram_range=(3, 5),
        max_features=80_000,
        min_df=2,
        sublinear_tf=True
    ))
])

model = Pipeline([
    ("features", text_features),
    ("classifier", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE))
])

model.fit(X_train, y_train)
print("Training complete.")


## 6. Evaluate the model

Accuracy alone may hide weak performance on less common genres. Review the per-class precision, recall, and F1-score as well. The reported scores will appear after you run the notebook with the dataset.


In [ ]:
# 6. Predictions and metrics
y_pred = model.predict(X_valid)
accuracy = accuracy_score(y_valid, y_pred)

print(f"Validation accuracy: {accuracy:.4f}")
report_dict = classification_report(y_valid, y_pred, zero_division=0, output_dict=True)
report_text = classification_report(y_valid, y_pred, zero_division=0)
print(report_text)

# Save reproducible evaluation outputs
with open(RESULTS_DIR / "classification_report.txt", "w", encoding="utf-8") as f:
    f.write(f"Validation accuracy: {accuracy:.6f}\n\n")
    f.write(report_text)

with open(RESULTS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump({"validation_accuracy": float(accuracy), "classification_report": report_dict}, f, indent=2)

print("Saved metrics to:", RESULTS_DIR.resolve())


In [ ]:
# 7. Confusion matrix for the 12 most common validation genres
top_genres = y_valid.value_counts().head(12).index.tolist()
mask = y_valid.isin(top_genres)
if mask.any():
    fig, ax = plt.subplots(figsize=(12, 10))
    ConfusionMatrixDisplay.from_predictions(
        y_valid[mask],
        pd.Series(y_pred, index=y_valid.index)[mask],
        labels=top_genres,
        xticks_rotation=90,
        cmap="Blues",
        ax=ax,
        colorbar=False
    )
    ax.set_title("Confusion Matrix — 12 Most Frequent Validation Genres")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=160, bbox_inches="tight")
    plt.show()


## 7. Try a custom plot summary

This is a single-label classifier. Real movies can have multiple genres, so interpret the output as the model's best single genre prediction.


In [ ]:
# 8. Test with your own plot summary
sample_plot = (
    "A young detective investigates a series of mysterious crimes in a quiet city, "
    "uncovering a dangerous conspiracy that puts the entire community at risk."
)
sample_prediction = model.predict([sample_plot])[0]
print("Sample plot:", sample_plot)
print("Predicted genre:", sample_prediction)


## 8. Save the trained model

The model artifact is written to `artifacts/`. It is excluded from Git by default to keep the repository lightweight; you can regenerate it by running this notebook.


In [ ]:
# 9. Save model and metadata
model_path = ARTIFACTS_DIR / "movie_genre_model.joblib"
joblib.dump({
    "model": model,
    "genres": sorted(df["genre"].unique().tolist()),
    "validation_accuracy": float(accuracy),
    "random_state": RANDOM_STATE
}, model_path)

print("Model saved to:", model_path.resolve())
print(f"Final validation accuracy: {accuracy:.4f}")


## 9. Conclusion

This project builds a baseline NLP classifier that predicts a movie's genre from its plot summary. TF-IDF represents text numerically and LinearSVC learns to distinguish genres.

**Before submitting:** run every notebook cell, check the saved evaluation files in `results/`, include screenshots of your own output, and report the measured validation score rather than an assumed score.

### Limitations and future improvements
- Some genres have fewer examples than others.
- The current setup predicts one genre per movie.
- Improvements could include hyperparameter tuning, class-specific error analysis, and a multi-label model for movies with several genres.
